# LGBM With Adding features Features

Test a small set of row-wise aggregate features on top of the current `LGBM` setup using the best parameters available from the Optuna run.

In [10]:
import sys

sys.path.append("../")

import json
from pathlib import Path

import numpy as np
import pandas as pd

In [11]:
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error, root_mean_squared_log_error
from sklearn.model_selection import KFold, cross_val_score, train_test_split

from src.features import add_rowwise_features
from src.loader import Loader
from src.modeling import build_lgbm_regressor

In [12]:
SEED = 42
TEST_SIZE = 0.33
CV = 5

In [13]:
loader = Loader()
df = loader.load("../data/processed_data.csv")
df.shape

(4459, 4732)

In [14]:
X = df.drop(columns="target")
y = df["target"]
y_log = np.log1p(y)

(X.shape, y.shape)

((4459, 4731), (4459,))

Only a minimal feature set is added here: row sparsity, overall row magnitude, and the average/spread of non-zero values.

In [16]:
X_aug = add_rowwise_features(X)
X_aug.shape

(4459, 4739)

In [17]:
X_aug_train, X_aug_test, y_train_raw, y_test_raw, y_train_log, y_test_log = train_test_split(
    X_aug,
    y,
    y_log,
    test_size=TEST_SIZE,
    random_state=SEED,
)

cv = KFold(n_splits=CV, shuffle=True, random_state=SEED)

In [ ]:
default_params = {
    "n_estimators": 500,
    "learning_rate": 0.03,
    "num_leaves": 31,
    "max_depth": 8,
    "min_child_samples": 20,
    "subsample": 0.8,
    "subsample_freq": 1,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.05,
    "reg_lambda": 0.05,
    "min_split_gain": 0.0,
}

best_params = default_params

{'n_estimators': 439,
 'learning_rate': 0.019291857433628438,
 'num_leaves': 32,
 'max_depth': 11,
 'min_child_samples': 5,
 'subsample': 0.9872554086372487,
 'subsample_freq': 5,
 'colsample_bytree': 0.7408418143236152,
 'reg_alpha': 0.025576812693216197,
 'reg_lambda': 0.718015922751547,
 'min_split_gain': 0.04545640466117268}

In [20]:
model = build_lgbm_regressor(best_params)
cv_scores = -cross_val_score(
    estimator=model,
    X=X_aug_train,
    y=y_train_log,
    cv=cv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
)

In [21]:
model.fit(X_aug_train, y_train_log)
y_pred_log = model.predict(X_aug_test)
y_pred = np.expm1(y_pred_log)
y_pred = np.clip(y_pred, 0, None)

results_df = pd.DataFrame(
    {
        "metric": ["n_features", "cv_rmsle_mean", "cv_rmsle_std", "holdout_rmsle", "holdout_rmse", "holdout_mae", "holdout_r2"],
        "value": [
            X_aug_train.shape[1],
            cv_scores.mean(),
            cv_scores.std(),
            root_mean_squared_log_error(y_test_raw, y_pred),
            root_mean_squared_error(y_test_raw, y_pred),
            mean_absolute_error(y_test_raw, y_pred),
            r2_score(y_test_raw, y_pred),
        ],
    }
)

results_df.style.format({"value": "{:,.4f}"})

,metric,value
0,n_features,"4,739.0000"
1,cv_rmsle_mean,1.3819
2,cv_rmsle_std,0.0405
3,holdout_rmsle,1.3931
4,holdout_rmse,"6,977,905.1656"
5,holdout_mae,"3,919,146.0388"
6,holdout_r2,0.2371


In [22]:
added_features = [
    col for col in X_aug.columns
    if col not in X.columns
]
added_features

['non_zero_count',
 'non_zero_ratio',
 'row_sum',
 'row_mean',
 'row_std',
 'row_max',
 'nz_mean',
 'nz_std']

In [23]:
ARTIFACTS_DIR = Path("../artifacts/rowwise_features_lgbm")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

results_df.to_csv(ARTIFACTS_DIR / "rowwise_metrics.csv", index=False)

summary = {
    "target_transform": "log1p",
    "primary_metric": "rmsle",
    "model_params": best_params,
    "added_features": added_features,
    "results": dict(zip(results_df["metric"], results_df["value"])),
}

with open(ARTIFACTS_DIR / "rowwise_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

## How To Read The Result

- Compare the `cv_rmsle_mean` and holdout `RMSLE` here against the reference result from `03_baseline.ipynb`.
- Accept the new features only if the CV result improves and the holdout result does not regress materially.
- If this wins, the next step is to rerun `Optuna` on the augmented feature space rather than adding many more aggregates.